# Indian Sign Language (ISL) Recognition System
## Phase 2: Custom Deep CNN Architecture Training, Evaluation & Edge Deployment

### 📌 Notebook Objectives
This notebook builds, trains, and evaluates a high-accuracy **Convolutional Neural Network (CNN)** for 26 Indian Sign Language alphabet gestures (A–Z).

Key Milestones:
1. **End-to-End Data Loading**: Efficient dataset streaming using `tf.keras.utils.image_dataset_from_directory`.
2. **Deep CNN Architecture Design**: 3 Conv-Pool blocks, Dropout regularization, Dense representation, Softmax 26-class classification.
3. **Training & Validation Monitoring**: 30 Epochs tracking accuracy and loss convergence.
4. **Comprehensive Evaluation**: Multi-class Confusion Matrix, Precision, Recall, F1-Score, Loss/Accuracy curves.
5. **Mobile & Edge Export**: Serializing to Keras `.keras` and optimizing into TensorFlow Lite (`.tflite`) for real-time edge devices.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Ensure output directories exist
os.makedirs("model", exist_ok=True)
os.makedirs("evaluation_results", exist_ok=True)

### 1. Dataset Configuration & Ingestion Pipeline
We configure standard image resolution ($224 \times 224$), batch size ($32$), and load the partitioned data.

In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 26
EPOCHS = 30

train_dir = "Dataset/Training"
val_dir = "Dataset/Validation"
test_dir = "Dataset/Testing"

print("--- Loading Datasets ---")
train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

val_data = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_data.class_names
print(f"\nIdentified {len(class_names)} Classes: {class_names}")

### 2. Custom CNN Architecture Design
The architecture is designed to balance high expressive capacity with lightweight inference latency suitable for edge deployment:
- **Rescaling Layer**: Maps $[0, 255]$ pixel inputs to $[0.0, 1.0]$.
- **Conv Block 1**: 32 filters ($3 \times 3$, ReLU) + MaxPool2D
- **Conv Block 2**: 64 filters ($3 \times 3$, ReLU) + MaxPool2D
- **Conv Block 3**: 128 filters ($3 \times 3$, ReLU) + MaxPool2D
- **Dense Classifier**: Flatten $\rightarrow$ Dense(128, ReLU) $\rightarrow$ Dropout(0.5) $\rightarrow$ Dense(26, Softmax)

In [3]:
normalization = layers.Rescaling(1.0 / 255)

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    normalization,
    
    # Feature extraction blocks
    layers.Conv2D(32, (3, 3), activation="relu", name="conv1"),
    layers.MaxPooling2D(name="pool1"),
    
    layers.Conv2D(64, (3, 3), activation="relu", name="conv2"),
    layers.MaxPooling2D(name="pool2"),
    
    layers.Conv2D(128, (3, 3), activation="relu", name="conv3"),
    layers.MaxPooling2D(name="pool3"),
    
    # Classification head
    layers.Flatten(name="flatten"),
    layers.Dense(128, activation="relu", name="dense_hidden"),
    layers.Dropout(0.5, name="dropout"),
    layers.Dense(NUM_CLASSES, activation="softmax", name="output_classes")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### 3. Model Training & Optimization
Training the model over 30 epochs with validation monitoring. (If pretrained model exists, we can load it directly).

In [4]:
keras_model_path = "model/custom_cnn.keras"

if os.path.exists(keras_model_path):
    print(f"Loading pre-trained model checkpoint from: {keras_model_path}")
    model = tf.keras.models.load_model(keras_model_path)
    print("Model loaded successfully!")
else:
    print(f"Training Custom CNN for {EPOCHS} epochs...")
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS
    )
    model.save(keras_model_path)
    print(f"Model saved to {keras_model_path}")

### 4. Evaluation Curves: Training vs Validation Accuracy & Loss
Visualizing convergence and verifying zero overfitting.

In [5]:
# Check if saved curves exist, or plot from history
acc_img_path = "evaluation_results/accuracy_curve.png"
loss_img_path = "evaluation_results/loss_curve.png"

if os.path.exists(acc_img_path) and os.path.exists(loss_img_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    acc_img = plt.imread(acc_img_path)
    loss_img = plt.imread(loss_img_path)
    
    axes[0].imshow(acc_img)
    axes[0].set_title("Training vs Validation Accuracy")
    axes[0].axis("off")
    
    axes[1].imshow(loss_img)
    axes[1].set_title("Training vs Validation Loss")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Accuracy and loss curve artifacts ready.")

### 5. Test Split Evaluation & Diagnostics
Evaluating the model against the independent, leak-free test dataset.

In [6]:
print("--- Evaluating Model on Test Split ---")
test_loss, test_accuracy = model.evaluate(test_data, verbose=1)
print(f"\n🎯 Final Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"🎯 Final Test Loss     : {test_loss:.4f}")

# Generate Predictions for Confusion Matrix and Classification Report
y_true = []
y_pred = []

for images, labels in test_data:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nClassification Report:")
report_str = classification_report(y_true, y_pred, target_names=class_names)
print(report_str)

### 6. Confusion Matrix Visualization
Inspecting class-by-class confusion matrix across all 26 alphabet signs (A–Z).

In [7]:
cm_img_path = "evaluation_results/confusion_matrix.png"

if os.path.exists(cm_img_path):
    plt.figure(figsize=(10, 8))
    img = plt.imread(cm_img_path)
    plt.imshow(img)
    plt.axis("off")
    plt.title("26-Class ISL Confusion Matrix Heatmap", fontsize=14)
    plt.show()
else:
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted Sign")
    plt.ylabel("Ground Truth Sign")
    plt.title("26-Class Confusion Matrix")
    plt.show()

### 7. Edge Deployment: TensorFlow Lite (.tflite) Conversion
We convert the trained Keras model into a lightweight TensorFlow Lite format for optimized execution on Raspberry Pi, mobile, or edge laptop devices.

In [8]:
tflite_path = "model/custom_cnn.tflite"

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

keras_size_mb = os.path.getsize(keras_model_path) / (1024 * 1024)
tflite_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)

print("=" * 50)
print(f"Keras Model Size   : {keras_size_mb:.2f} MB")
print(f"TFLite Model Size  : {tflite_size_mb:.2f} MB")
print(f"Storage Reduction  : {(1 - tflite_size_mb / keras_size_mb) * 100:.1f}%")
print("=" * 50)
print("✅ Edge model export successfully verified!")